# bio-trajectory-eval walkthrough

This notebook shows how the trajectory dataset is structured and how a single model run is scored. It is meant for a reviewer who wants to inspect the method before reading the code.

The dataset uses safe proxy domains only. The items test workflow shape, not dangerous biological content.

The live model cells require either `ANTHROPIC_API_KEY` or `OPENAI_API_KEY`. Without a key, the notebook still validates the dataset and shows the example trajectories.

In [ ]:
from collections import Counter
from pathlib import Path
import os, sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from bio_trajectory_eval.schema import load_trajectories
from bio_trajectory_eval.harness import HarnessConfig, run_trajectory
from bio_trajectory_eval.adapters.anthropic import AnthropicAdapter
from bio_trajectory_eval.adapters.openai import OpenAIAdapter

In [ ]:
trajectories = load_trajectories(ROOT / 'data' / 'trajectories.jsonl')
print(f'loaded trajectories: {len(trajectories)}')
print(Counter(t.difficulty.value for t in trajectories))
print(Counter(t.proxy_domain.value for t in trajectories))

## Three trajectories

The examples below show one item per tier. The transcript includes only user turns so the intended trajectory shape is visible before any model behavior is introduced.

In [ ]:
for tier in ['tier_1', 'tier_2', 'tier_3']:
    item = next(t for t in trajectories if t.difficulty.value == tier)
    print('\n' + '=' * 72)
    print(item.id, item.difficulty.value, item.proxy_domain.value)
    print(item.shape_of_real_workflow)
    print(item.trajectory_level_concern)
    for turn in item.turns:
        print(f'User {turn.turn_index}: {turn.user_message}')

## Single live Tier 2 run

This cell runs one Tier 2 trajectory if a key is available. A trajectory-aware safeguard should redirect once the accumulated direction of the conversation becomes clear.

In [ ]:
tier2 = [t for t in trajectories if t.difficulty.value == 'tier_2']
if os.environ.get('ANTHROPIC_API_KEY'):
    adapter = AnthropicAdapter('claude-sonnet-4-6')
elif os.environ.get('OPENAI_API_KEY'):
    adapter = OpenAIAdapter('gpt-4.1')
else:
    adapter = None

if adapter is None:
    print('No API key found. Skipping live model run.')
else:
    result = run_trajectory(tier2[0], adapter, HarnessConfig())
    print(result.trajectory_id, result.trajectory_label.value)
    for turn in result.turns:
        print('\nUSER:', turn.user_message)
        print('MODEL:', turn.model_response)
        print('LABEL:', turn.handling_label.value)

## Full Tier 2 run

The full run is intentionally explicit. It writes JSONL to `results/` and prints the trajectory labels. On a fresh checkout this is the first place a reviewer should look for failure modes such as `compliant_throughout` on a Tier 2 item.

In [ ]:
if adapter is None:
    print('No API key found. Skipping full Tier 2 run.')
else:
    results = [run_trajectory(t, adapter, HarnessConfig()) for t in tier2]
    counts = Counter(r.trajectory_label.value for r in results)
    print(counts)
    for r in results:
        if r.trajectory_label.value == 'compliant_throughout':
            print('\nPotential failure mode:', r.trajectory_id)
            for turn in r.turns:
                print('USER:', turn.user_message)
                print('MODEL:', turn.model_response[:500])
            break

## What this does and does not show

The notebook shows whether a model changes behavior across safe proxy trajectories. It does not measure dangerous biological capability, and it does not validate performance on gated dangerous-adjacent content.

To scale this method, the dataset needs expert review, a private access-controlled tier, and human review of disagreements between rules and the judge.